# 04.1 Variables and Assignment

Almost every tutorial says a variable is "a box that holds a value". That model
is **wrong in Python**, and it is the reason two lists can appear to change when
you only touched one.

This chapter replaces it with the model Python actually uses. Get this right and
a whole category of baffling bugs simply stops happening.

## Theory

### The box model, and why it fails

In C, a variable genuinely is a box — a fixed piece of memory that holds a value:

```c
int x = 5;      // reserve memory, put 5 in it
x = 10;         // overwrite that same memory with 10
```

In Python, `x = 5` does something different:

1. Create an **integer object** with the value 5, somewhere in memory
2. Make the **name** `x` refer to that object

The name is a **label**, not a container. Assignment does not copy a value into a
box — it points a label at an object.

### Objects have three properties

Every Python object has:

<table>
<tr><th>Property</th><th>Meaning</th><th>How to see it</th></tr>
<tr><td><b>Identity</b></td><td>Where it lives — never changes</td><td><code>id(x)</code></td></tr>
<tr><td><b>Type</b></td><td>What it is — never changes</td><td><code>type(x)</code></td></tr>
<tr><td><b>Value</b></td><td>What it holds — <i>may</i> change</td><td><code>x</code></td></tr>
</table>

Identity and type are fixed for an object's entire life. Only the value can
change, and only for some types (Chapter 04.4).

### What rebinding actually does

```python
x = 5      # x -> object(5)
x = 10     # x -> object(10)     the 5 object is untouched
```

The second line did **not** modify the integer 5. It created a new object and
repointed the label. The original 5 still exists, unchanged — if nothing else
refers to it, Python will eventually reclaim it (04.6).

This matters because it means **assignment never modifies an object**. It only
changes what a name points to.

### Names live in namespaces

A namespace is a dictionary mapping names to objects. When you write `x = 5`,
Python adds the key `"x"` to a namespace with the integer object as its value.
You can inspect this directly with `globals()` and `locals()`.

In [ ]:
# Assignment binds a NAME to an OBJECT. Watch the identity.
x = 5

print("After x = 5")
print("   value:", x)
print("   type: ", type(x).__name__)
print("   id:   ", id(x))

# Rebinding points the name at a DIFFERENT object.
x = 10

print("")
print("After x = 10")
print("   value:", x)
print("   id:   ", id(x), "<- a different object entirely")

print("")
print("The integer 5 was never modified. A new object was created,")
print("and the label moved to it.")

## Names are keys in a namespace dictionary

`globals()` returns the actual dictionary Python uses. Assignment is literally a
dictionary insertion.

In [ ]:
# Create a name and then find it in the namespace dictionary.
course_name = "Python Learning"

# globals() is the real namespace - not a copy.
namespace = globals()

print("Is 'course_name' a key in globals()?", "course_name" in namespace)
print("What is stored under that key?", repr(namespace["course_name"]))

# Because it is a real dictionary, writing to it creates a variable.
namespace["created_via_dictionary"] = 42

# The name now exists as if we had assigned it normally.
print("")
print("created_via_dictionary =", created_via_dictionary)

print("")
print("This is what assignment does underneath: a dictionary insert.")
print("You would never write it this way - but it shows the mechanism.")

## Under the hood: the bytecode of assignment

The compiled instructions show the two steps plainly: load the object, then
store the name.

In [ ]:
import dis

print("=== x = 5 ===")
dis.dis(compile("x = 5", "<demo>", "exec"))

print("")
print("=== x = y ===")
dis.dis(compile("x = y", "<demo>", "exec"))

print("")
print("Read the second listing: LOAD_NAME y, then STORE_NAME x.")
print("Nothing is copied. Both names end up pointing at one object.")

## The consequence: `a = b` does not copy

This is the single most important practical result of the label model. For
**immutable** types you never notice. For **mutable** types it changes
everything.

In [ ]:
# Two names, one list object.
original = [1, 2, 3]
second_name = original

print("original     :", original)
print("second_name  :", second_name)
print("same object? :", original is second_name)
print("same id?     :", id(original) == id(second_name))

# Modifying through ONE name is visible through the OTHER,
# because there is only one object.
second_name.append(4)

print("")
print("After second_name.append(4):")
print("   original    :", original, "<- changed, though we never touched it")
print("   second_name :", second_name)

print("")
print("There was never a second list. Two labels, one object.")

In [ ]:
# The same code with an IMMUTABLE type looks like it behaves differently.
original_number = 10
second_number = original_number

print("Before:")
print("   original_number:", original_number)
print("   second_number:  ", second_number)
print("   same object?    ", original_number is second_number)

# This looks like modification, but it is REBINDING.
second_number = second_number + 1

print("")
print("After second_number = second_number + 1:")
print("   original_number:", original_number, "<- unchanged")
print("   second_number:  ", second_number)
print("   same object?    ", original_number is second_number)

print("")
print("The rule is identical in both cases. The difference is that an int")
print("cannot be modified in place, so + had to build a new object.")
print("Mutability is covered in 04.4.")

## Assignment forms

Python has several ways to bind names. All of them follow the same rule.

In [ ]:
# 1. Simple assignment.
single = 1

# 2. Multiple assignment - one object, several names.
first = second = third = []

print("1. Chained assignment:")
print("   all three names point at one object?",
      first is second is third)

# The trap: they share the object.
first.append("added via first")
print("   after first.append(...), third is:", third)

# 3. Tuple unpacking - several objects, several names.
a_value, b_value, c_value = 1, 2, 3
print("")
print("2. Unpacking:", a_value, b_value, c_value)

# 4. Swapping, without a temporary variable.
left, right = "L", "R"
left, right = right, left
print("3. After swap:", left, right)

# 5. Extended unpacking - the star collects the rest.
head, *middle, tail = [1, 2, 3, 4, 5]
print("4. head:", head, " middle:", middle, " tail:", tail)

# 6. Augmented assignment.
counter = 10
counter += 5
print("5. After counter += 5:", counter)

# 7. Annotated assignment - a type hint, checked by tools not by Python.
total_price: float = 99.99
print("6. Annotated:", total_price)

### Why the swap works

`left, right = right, left` is not two statements. The right-hand side is
evaluated **completely first**, building a tuple, and only then is it unpacked.
The bytecode makes this obvious.

In [ ]:
import dis

print("=== left, right = right, left ===")
dis.dis(compile("left, right = right, left", "<demo>", "exec"))

print("")
print("Both values are loaded onto the stack BEFORE either name is stored.")
print("ROT_TWO / SWAP just reorders them. No temporary variable needed.")
print("")
print("This is why the naive C-style version needs a temp, and Python does not:")
print("   temp = left; left = right; right = temp     <- three statements")
print("   left, right = right, left                   <- one statement")

## Augmented assignment is not always what it seems

`x += 1` looks like pure shorthand for `x = x + 1`. For immutable types it is.
For mutable types it is **different**, because `+=` tries to modify in place
first.

This is one of Python's genuine subtleties, and it catches experienced people.

In [ ]:
# With a LIST, += modifies the existing object in place.
list_a = [1, 2]
list_b = list_a
id_before = id(list_a)

list_a += [3]

print("LIST with +=")
print("   list_a:", list_a)
print("   list_b:", list_b, "<- also changed")
print("   same object as before?", id(list_a) == id_before)

# With a LIST, = + builds a NEW object instead.
list_c = [1, 2]
list_d = list_c
id_before = id(list_c)

list_c = list_c + [3]

print("")
print("LIST with = +")
print("   list_c:", list_c)
print("   list_d:", list_d, "<- unchanged")
print("   same object as before?", id(list_c) == id_before)

print("")
print("SAME SYNTAX, DIFFERENT RESULT:")
print("   x += y   calls __iadd__ - modifies in place when possible")
print("   x = x+y  calls __add__  - always builds a new object")

In [ ]:
# With an immutable type, += cannot modify in place, so both forms match.
number_a = 10
id_before = id(number_a)
number_a += 5

print("INT with +=")
print("   value:", number_a)
print("   same object?", id(number_a) == id_before, "<- had to create a new one")

# Strings behave the same way - they are immutable.
text = "hello"
id_before = id(text)
text += " world"

print("")
print("STR with +=")
print("   value:", repr(text))
print("   same object?", id(text) == id_before)

print("")
print("So for immutable types += is genuinely just shorthand.")
print("For mutable types it is a different operation. Know which you have.")

## Deleting a name

`del` removes the **name**, not the object. The object survives as long as
something else refers to it.

In [ ]:
# Two names pointing at one list.
kept = [1, 2, 3]
removed = kept

print("Before del:")
print("   kept   :", kept)
print("   removed:", removed)

# del removes the NAME from the namespace.
del removed

print("")
print("After del removed:")
print("   'removed' still a name?", "removed" in globals())
print("   kept:", kept, "<- the object is fine")

# Using the deleted name is now a NameError.
try:
    removed
except NameError as error:
    print("   accessing it:", error)

print("")
print("del unbinds a name. The object is only reclaimed when NOTHING")
print("refers to it any more - covered in 04.6.")

## Assignment before definition

Python evaluates the right-hand side first, so a name must already exist before
you can read it.

In [ ]:
# Reading a name that was never bound.
try:
    print(never_assigned)
except NameError as error:
    print("Reading an unbound name:", error)

# A subtler case: reading a name on the line that defines it.
try:
    undefined_so_far = undefined_so_far + 1
except NameError as error:
    print("Self-referencing assignment:", error)

print("")
print("WHY: the right-hand side is evaluated BEFORE the name is bound.")
print("So `x = x + 1` requires x to already exist.")

# The fix is to initialise first.
running_total = 0
running_total = running_total + 1
print("")
print("Initialised first, then updated:", running_total)

## Takeaways

1. A variable is a **label pointing at an object**, not a box holding a value.
2. Every object has a fixed **identity** and **type**, and a value that may or
   may not be changeable.
3. Assignment **never modifies an object** — it rebinds a name.
4. Names live in namespace **dictionaries**; `x = 5` is effectively a dict insert.
5. `a = b` **copies nothing** — you get two names for one object.
6. `left, right = right, left` works because the right side is fully evaluated
   into a tuple first.
7. `x += y` and `x = x + y` are the **same for immutable types** and **different
   for mutable ones** — `+=` modifies in place when it can.
8. `del` removes a name, not an object.

## Try it yourself

1. Assign `a = [1, 2]` then `b = a`. Append to `b` and print `a`. Explain why.
2. Repeat with `a = (1, 2)`. Why can you not append, and what changes?
3. Run `id(x)` before and after `x += 1` for an int and for a list. Which keeps
   its identity?
4. Use `globals()` to create a variable without an assignment statement.
5. Predict the output, then check: `a = b = []; a.append(1); print(b)`.